# 04 — Results Summary

Final comparison of Naive, ARIMA(1,0,2), and Prophet across forecast horizons of 1, 5, and 21 trading days.
Results are loaded from the pre-computed walk-forward backtest (36 folds, test_size=756, step=21).

In [ ]:
import sys, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

# Load backtest results
summary = pd.read_csv('../outputs/predictions/backtest_summary.csv')
print(f"Loaded {len(summary)} rows from backtest_summary.csv")
summary

## 1. Final Comparison Table

Mean metrics across 36 expanding-window folds. Lower RMSE/MAE/MAPE is better; higher directional accuracy is better. Bold = best per horizon.

In [ ]:
MODEL_ORDER = ['Naive', 'ARIMA', 'Prophet']
MODEL_LABELS = {'Naive': 'Naive (Persistence)', 'ARIMA': 'ARIMA(1,0,2)', 'Prophet': 'Prophet'}

# Pivot for display
pivot = summary.pivot(index='model', columns='horizon')
pivot = pivot.reindex(MODEL_ORDER)

# Pretty display
display_rows = []
for model in MODEL_ORDER:
    row = {'Model': MODEL_LABELS[model]}
    for h in [1, 5, 21]:
        row[f'RMSE h={h}'] = round(summary.query(f"model=='{model}' and horizon=={h}")['rmse'].values[0], 3)
        row[f'MAE h={h}']  = round(summary.query(f"model=='{model}' and horizon=={h}")['mae'].values[0], 3)
        row[f'DirAcc h={h}'] = f"{summary.query(f\"model=='{model}' and horizon=={h}\")['dir_acc'].values[0]:.1%}"
    display_rows.append(row)

display_df = pd.DataFrame(display_rows).set_index('Model')
display_df

## 2. RMSE by Horizon — Bar Chart

In [ ]:
from IPython.display import Image
Image('../figures/backtest_metrics_comparison.png')

## 3. Forecast vs Actual — Walk-Forward Window

In [ ]:
Image('../figures/backtest_forecast_vs_actual.png')

## 4. Failure Mode Analysis

### 4a. Naive — zero directional accuracy by construction

In [ ]:
# Naive directional accuracy is 0 because it always forecasts flat (zero change).
# The directional_accuracy metric excludes flat actual moves (mask = actual_dirs != 0).
# A flat forecast yields pred_dir=0, which never matches actual_dir ∈ {-1, +1}.
# This is not a bug — it correctly captures that persistence is a useless direction signal.

naive_summary = summary[summary['model'] == 'Naive']
print("Naive directional accuracy by horizon:")
print(naive_summary[['horizon', 'dir_acc']].to_string(index=False))
print()
print("Interpretation: a flat forecast (last value repeated) predicts zero change at every step.")
print("Since actual VIX moves are almost never exactly zero, dir_acc = 0% is mathematically exact.")

### 4b. ARIMA — directional accuracy decay with horizon

In [ ]:
arima_dir = summary[summary['model'] == 'ARIMA'][['horizon', 'dir_acc']].set_index('horizon')

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot([1, 5, 21], arima_dir.loc[[1, 5, 21], 'dir_acc'] * 100,
        'o-', color='#2B6CB0', linewidth=2, markersize=8, label='ARIMA(1,0,2)')
ax.axhline(50, color='gray', linestyle='--', linewidth=1, label='Random baseline (50%)')
ax.axhline(summary[summary['model'] == 'Prophet']['dir_acc'].mean() * 100,
           color='#E07070', linestyle=':', linewidth=1, label='Prophet (avg)')
ax.set_xlabel('Forecast horizon (trading days)', fontsize=10)
ax.set_ylabel('Directional Accuracy (%)', fontsize=10)
ax.set_title('ARIMA Directional Accuracy vs Horizon', fontsize=11)
ax.set_xticks([1, 5, 21])
ax.set_ylim(40, 80)
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
fig.tight_layout()
fig.savefig('../figures/arima_dir_acc_vs_horizon.png', dpi=150)
plt.show()

print("ARIMA directional accuracy:")
for h in [1, 5, 21]:
    da = arima_dir.loc[h, 'dir_acc']
    print(f"  h={h:2d}d : {da:.1%}  ({'above' if da > 0.5 else 'at/below'} 50% random)")
print()
print("At h=21, ARIMA direction accuracy collapses to ~50% — a coin flip.")
print("This is expected: the AR(1)=0.984 persistence structure carries directional information")
print("only at short horizons. Over 21 days, mean-reversion pulls all forecasts toward the")
print("same long-run mean (~19.3), eliminating directional signal.")

### 4c. Prophet — trend decomposition mismatch with mean-reverting VIX

In [ ]:
# Quantify Prophet's systematic bias: RMSE 5x worse than ARIMA
prophet_rmse = summary[summary['model'] == 'Prophet']['rmse'].values
arima_rmse   = summary[summary['model'] == 'ARIMA']['rmse'].values
naive_rmse   = summary[summary['model'] == 'Naive']['rmse'].values
horizons = [1, 5, 21]

print("Prophet vs ARIMA RMSE ratio:")
for h, pr, ar in zip(horizons, prophet_rmse, arima_rmse):
    print(f"  h={h:2d}d : Prophet {pr:.3f} vs ARIMA {ar:.3f}  →  {pr/ar:.1f}x worse")

print()
print("Root cause: Prophet fits a piecewise linear trend to VIX history.")
print("During 2023-2026 test period (low-vol regime, VIX largely 12-20), the trend")
print("extracted from training data (inc. COVID spike at 85) anchors forecasts too high.")
print("This is a structural incompatibility — VIX is stationary/mean-reverting,")
print("not trend-stationary. Prophet's design assumption is violated.")

### 4d. All models — spike onset not captured

In [ ]:
from src.vix_forecasting.data.preprocessing import load_raw, build_series

vix = build_series(load_raw(Path('../data/raw/vix_raw.csv')))

# Look at large VIX spikes in the test window (last 756 obs = ~3 years)
test_vix = vix.iloc[-756:]
daily_chg = test_vix.diff().dropna()
large_spikes = daily_chg[daily_chg > 3].sort_values(ascending=False)

print(f"VIX spikes > 3 pts in test window ({test_vix.index[0].date()} to {test_vix.index[-1].date()}):")
print(large_spikes.head(10).to_string())
print()
print("These abrupt jumps are driven by exogenous macro/geopolitical events that are")
print("not linearly predictable from past VIX values. All linear models (Naive, ARIMA,")
print("Prophet) systematically under-forecast spike magnitude. The AR(1) structure of")
print("ARIMA provides the fastest *recovery* tracking (best RMSE post-spike) but cannot")
print("predict spike *onset* without exogenous regressors (e.g. economic surprise indices,")
print("options flow, macro event calendars).")

## 5. Summary and Verdict

| Model | Strengths | Weaknesses |
|-------|-----------|-----------|
| **Naive** | Zero parameters, interpretable floor | 0% directional accuracy; no mean reversion |
| **ARIMA(1,0,2)** | Best RMSE/MAE all horizons; 69% dir acc at h=1; fast inference | Dir acc collapses at h=21; cannot predict spike onset |
| **Prophet** | Better than Naive on direction; handles calendar effects | 5× worse RMSE; trend decomposition incompatible with stationarity |

**ARIMA(1,0,2) is the recommended model** for VIX point forecasting under the classical linear framework.

**Extensions for future work:**
- GARCH(1,1) on VIX returns: captures volatility-of-volatility clustering
- ARIMAX with macro regressors (e.g. credit spreads, put/call ratio)
- Regime-switching models: separate AR dynamics for calm vs crisis regimes
- LSTM/TCN with attention: for spike prediction with exogenous inputs

---
## 6. Residual Diagnostics — ARIMA(1,0,2)

Diagnostics on the model fitted to the full 9,183-observation series.
All three tests reject their null hypotheses — findings reported honestly below.

In [ ]:
import json

with open('../outputs/diagnostics/residual_diagnostics.json') as f:
    diag = json.load(f)

diag_display = pd.DataFrame([
    {
        'Test': 'Ljung-Box (lag=20)',
        'Statistic': diag['ljung_box_stat'],
        'p-value': f"{diag['ljung_box_pvalue']:.3e}" if diag['ljung_box_pvalue'] < 0.001 else diag['ljung_box_pvalue'],
        'H0': 'no autocorrelation',
        'Conclusion': 'REJECT — residual autocorrelation remains',
    },
    {
        'Test': 'Jarque-Bera',
        'Statistic': f"{diag['jarque_bera_stat']:,.0f}",
        'p-value': f"{diag['jarque_bera_pvalue']:.3e}" if diag['jarque_bera_pvalue'] < 0.001 else diag['jarque_bera_pvalue'],
        'H0': 'normally distributed',
        'Conclusion': f"REJECT — skewness={diag['jarque_bera_detail']['skewness']}, excess kurtosis={diag['jarque_bera_detail']['excess_kurtosis']}",
    },
    {
        'Test': 'ARCH-LM (lag=12)',
        'Statistic': diag['arch_lm_stat'],
        'p-value': f"{diag['arch_lm_pvalue']:.3e}" if diag['arch_lm_pvalue'] < 0.001 else diag['arch_lm_pvalue'],
        'H0': 'no ARCH effects',
        'Conclusion': 'REJECT — variance clustering present',
    },
])
diag_display.set_index('Test')

In [ ]:
from IPython.display import Image
print("Q-Q Plot:")
display(Image('../figures/arima_residual_qq.png'))
print("Residual ACF:")
display(Image('../figures/arima_residual_acf.png'))

## 7. Log-Scale Robustness Check

ARIMA(1,0,2) refitted on log(VIX), same 36 folds, same horizons.
Log forecasts exponentiated back to VIX units before computing metrics.

In [ ]:
log_summary = pd.read_csv('../outputs/predictions/backtest_summary_log_vix.csv')
arima_level = summary[summary['model'] == 'ARIMA'].copy()

rows = []
for h in [1, 5, 21]:
    lvl = arima_level[arima_level['horizon'] == h].iloc[0]
    lg  = log_summary[log_summary['horizon'] == h].iloc[0]
    for metric in ['rmse', 'mae', 'mape', 'dir_acc']:
        rows.append({
            'Horizon': f'h={h}',
            'Metric': metric.upper(),
            'Level ARIMA': round(lvl[metric], 4),
            'Log-scale ARIMA': round(lg[metric], 4),
            'Delta': round(lg[metric] - lvl[metric], 4),
        })

comp_df = pd.DataFrame(rows)
comp_df['Better'] = comp_df.apply(
    lambda r: 'Log' if (r['Delta'] < 0 and r['Metric'] != 'DIR_ACC') or
                       (r['Delta'] > 0 and r['Metric'] == 'DIR_ACC')
              else 'Level' if r['Delta'] != 0 else 'Tie',
    axis=1
)
comp_df

In [ ]:
wins = comp_df['Better'].value_counts()
print("Win count across all horizon x metric combinations:")
print(wins.to_string())
print()
print("Verdict: differences are negligible (<2% RMSE, <3pp dir-acc).")
print("Level-scale ARIMA remains the recommended model for interpretability.")
print("Log transform does not meaningfully stabilize variance — spikes are discrete events,")
print("not proportional variance growth, so GARCH would address this more directly.")

## 8. Rolling Volatility of VIX

In [ ]:
Image('../figures/vix_rolling_volatility_by_year.png')

## 9. Practical Questions Answered

| Question | h=1 | h=5 | h=21 |
|----------|-----|-----|------|
| RMSE (VIX pts) | 1.09 | 1.81 | 3.05 |
| MAE (VIX pts) | 1.09 | 1.56 | 2.53 |
| MAPE | 5.93% | 8.52% | 13.54% |
| Directional accuracy | 69.4% | 56.1% | 50.4% |

**"Can we predict VIX 1 day ahead?"** Yes, modestly. 69% directional accuracy and ~1 VIX pt error. Useful for short-term risk sizing and delta hedge adjustments; not a trading signal on its own.

**"Can we predict VIX 5 days ahead?"** Weakly. Directional accuracy drops to 56% — above random, but barely. Useful for weekly rebalancing or options-roll decisions. Should not be relied on in uncertain regimes.

**"Can we predict VIX 21 days ahead?"** Barely better than guessing (50.4% directional accuracy, RMSE=3.05). The forecast converges toward the long-run mean regardless of current VIX. Not useful for directional decisions; only useful as a "no new shocks" baseline.

**"Can the model anticipate spike onsets?"** No. All models fail here. Spikes are driven by exogenous events not encoded in past VIX values. The ARCH-LM result confirms the model knows variance clusters — but not *when* the next cluster begins. Spike prediction requires exogenous regressors (credit spreads, macro surprise indices, options flow).